# Historical experiment results

Reloads verified exports written by `lotto research holdout` and `lotto research report`. It never fits, tunes or reruns the holdout. Set `OUTPUT` to a frozen experiment's `holdout` directory under ignored `data/experiments/`.

In [ ]:
import hashlib
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

OUTPUT = Path("data/experiments/<protocol_digest>/frozen/<identity>/holdout")

In [ ]:
# Refuse to plot anything whose hash differs from the manifest.
manifest = json.loads((OUTPUT / "manifest.json").read_text())
for name, digest in manifest.items():
    assert hashlib.sha256((OUTPUT / name).read_bytes()).hexdigest() == digest, name
metrics = json.loads((OUTPUT / "metrics.json").read_text())
findings = json.loads((OUTPUT / "findings.json").read_text())
print(findings["verdict"])

In [ ]:
comparison = pd.read_csv(OUTPUT / "model_comparison.csv")
comparison

In [ ]:
results = pd.read_csv(OUTPUT / "evaluation_results.csv")
one_line = results[results.budget == 1]
histogram = one_line.groupby(["policy", "matched_mains"]).size().unstack(fill_value=0)
histogram.T.plot.bar(title="One-line match histogram (holdout)")
plt.xlabel("main numbers matched")
plt.ylabel("draws")
plt.show()

In [ ]:
calibration = pd.DataFrame(findings["calibration"]).dropna()
ax = calibration.plot.scatter(
    x="mean_probability", y="observed_rate", title="Calibration"
)
ax.plot(
    [0, calibration.mean_probability.max()],
    [0, calibration.mean_probability.max()],
    ls="--",
)
plt.show()